# ODIR-5K — Шаг 4: Grad-CAM визуализация
**Дипломная работа:** Автоматизация диагностики офтальмологических заболеваний

Grad-CAM показывает **на какие области снимка смотрит модель** при постановке диагноза.
Тепловая карта накладывается прямо на снимок глазного дна.

## Шаг 1 — Подключение и загрузка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive подключён')

In [ ]:
!pip install -q albumentations

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DRIVE_DIR  = Path('/content/drive/MyDrive/diploma_odir')
GRADCAM_DIR = Path('/content/gradcam_results')
GRADCAM_DIR.mkdir(exist_ok=True)

with open(DRIVE_DIR / 'config.json', encoding='utf-8') as f:
    config = json.load(f)

CLASSES     = config['classes']
CLASS_NAMES = config['class_names']
IMG_SIZE    = config['img_size']
THRESHOLDS  = config.get('thresholds', {c: 0.5 for c in CLASSES})

print(f'Device: {DEVICE}')
print(f'Классы: {CLASSES}')
print('✅ Готово')

## Шаг 2 — Загрузка модели

In [ ]:
def build_model(num_classes=8):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

checkpoint = torch.load(
    DRIVE_DIR / 'best_efficientnet_b0.pth',
    map_location=DEVICE
)
model = build_model(len(CLASSES)).to(DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f'✅ Модель загружена (epoch={checkpoint["epoch"]}, val_auc={checkpoint["val_auc"]:.4f})')

## Шаг 3 — Grad-CAM реализация
Используем последний сверточный слой EfficientNet — `features[-1]`

In [ ]:
class GradCAM:
    """
    Grad-CAM для EfficientNet-B0.
    Цепляется к последнему сверточному блоку (features[-1]).
    """
    def __init__(self, model):
        self.model      = model
        self.gradients  = None
        self.activations = None
        # EfficientNet: последний блок features
        target_layer = model.features[-1]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, image_tensor, class_idx):
        """
        image_tensor : (1, 3, H, W) на DEVICE
        class_idx    : индекс класса (0..7)
        Возвращает: heatmap (H, W) в диапазоне [0, 1]
        """
        self.model.zero_grad()
        output = self.model(image_tensor)          # (1, 8)
        score  = output[0, class_idx]
        score.backward()

        # Усредняем градиенты по пространственным измерениям
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        cam     = (weights * self.activations).sum(dim=1)        # (1, H, W)
        cam     = F.relu(cam).squeeze(0).cpu().numpy()           # (H, W)

        # Нормализация в [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam


def overlay_heatmap(original_img, cam, alpha=0.45):
    """
    Накладывает тепловую карту на оригинальный снимок.
    original_img : np.array (H, W, 3) uint8
    cam          : np.array (h, w) float [0,1]
    Возвращает: np.array (H, W, 3) uint8
    """
    h, w = original_img.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam_resized), cv2.COLORMAP_JET
    )
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (alpha * heatmap + (1 - alpha) * original_img).astype(np.uint8)
    return overlay, cam_resized


def get_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])


gradcam = GradCAM(model)
transforms = get_transforms()
print('✅ Grad-CAM готов')

## Шаг 4 — Функция визуализации одного снимка

In [ ]:
def predict_and_visualize(image_path, save_path=None, title=None):
    """
    Предсказывает диагноз и строит Grad-CAM для всех обнаруженных классов.
    """
    # Загрузка и препроцессинг
    orig_img = np.array(Image.open(image_path).convert('RGB'))
    tensor   = transforms(image=orig_img)['image'].unsqueeze(0).to(DEVICE)

    # Предсказание
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.sigmoid(logits).cpu().numpy()[0]

    # Определяем обнаруженные классы
    detected = [
        (i, cls)
        for i, cls in enumerate(CLASSES)
        if probs[i] >= THRESHOLDS.get(cls, 0.5)
    ]
    if not detected:
        # Берём класс с максимальной вероятностью
        i = int(np.argmax(probs))
        detected = [(i, CLASSES[i])]

    n_detected = len(detected)
    fig_width  = 4 + n_detected * 4
    fig, axes  = plt.subplots(1, n_detected + 1, figsize=(fig_width, 4.5))
    if n_detected == 0:
        axes = [axes]

    # Оригинальный снимок
    axes[0].imshow(orig_img)
    axes[0].set_title('Оригинал', fontsize=11)
    axes[0].axis('off')

    # Grad-CAM для каждого обнаруженного класса
    for plot_idx, (class_idx, cls) in enumerate(detected):
        # Нужен grad — делаем forward с grad_enabled
        tensor_grad = tensor.clone().requires_grad_(True)
        cam = gradcam.generate(tensor_grad, class_idx)
        overlay, _ = overlay_heatmap(orig_img, cam, alpha=0.45)

        prob = probs[class_idx]
        ax   = axes[plot_idx + 1]
        ax.imshow(overlay)
        ax.set_title(
            f'{CLASS_NAMES[cls]}\n{prob*100:.1f}%',
            fontsize=11,
            color='#993C1D' if prob >= 0.7 else '#BA7517'
        )
        ax.axis('off')

    # Общий заголовок
    labels_str = ', '.join([CLASS_NAMES[c] for _, c in detected])
    suptitle = title or f'Диагноз: {labels_str}'
    plt.suptitle(suptitle, fontsize=12, y=1.02)

    # Таблица вероятностей снизу
    prob_str = '  |  '.join([
        f"{CLASS_NAMES[cls]}: {probs[i]*100:.1f}%"
        for i, cls in enumerate(CLASSES)
    ])
    plt.figtext(0.5, -0.04, prob_str, ha='center', fontsize=8,
                color='gray', wrap=True)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'  Сохранено → {save_path}')

    plt.show()
    plt.close()

    # Печатаем результат
    print(f'\nРезультат предсказания:')
    print(f'{"Заболевание":<20} {"Вероятность":>12}  {"Статус":>12}')
    print('-' * 48)
    for i, cls in enumerate(CLASSES):
        detected_flag = probs[i] >= THRESHOLDS.get(cls, 0.5)
        marker = '✅ ОБНАРУЖЕНО' if detected_flag else '—'
        print(f'{CLASS_NAMES[cls]:<20} {probs[i]*100:>10.1f}%  {marker}')

    return probs

print('✅ Функция готова')

## Шаг 5 — Grad-CAM на снимках из датасета
Скачиваем датасет и берём по одному снимку каждого класса

In [ ]:
# Скачиваем датасет если нужно
import os
if not Path('/content/data').exists():
    print('Скачиваем датасет...')
    !kaggle datasets download -d andrewmvd/ocular-disease-recognition-odir5k \
        -p /content/data --unzip -q
    print('✅ Датасет скачан')
else:
    print('✅ Датасет уже есть')

# Находим папку со снимками
train_dirs = list(Path('/content/data').rglob('Training Images'))
TRAIN_IMG_DIR = train_dirs[0]
print(f'Снимки: {TRAIN_IMG_DIR}')

In [ ]:
# Исправляем пути в test.csv
test_df = pd.read_csv(DRIVE_DIR / 'test.csv')
test_df['filepath'] = test_df['filename'].apply(
    lambda fn: str(TRAIN_IMG_DIR / fn)
)

# Берём по 1 снимку каждого класса для демонстрации
samples = {}
for cls in CLASSES:
    subset = test_df[test_df[cls] == 1]
    if len(subset) > 0:
        row = subset.sample(1, random_state=42).iloc[0]
        if Path(row['filepath']).exists():
            samples[cls] = row['filepath']

print(f'Найдено примеров: {len(samples)}')
for cls, path in samples.items():
    print(f'  {CLASS_NAMES[cls]}: {Path(path).name}')

In [ ]:
# Генерируем Grad-CAM для каждого класса
print('Генерируем Grad-CAM визуализации...\n')

for cls, img_path in samples.items():
    print(f'--- {CLASS_NAMES[cls]} ---')
    save_path = GRADCAM_DIR / f'gradcam_{cls}_{CLASS_NAMES[cls]}.png'
    predict_and_visualize(
        image_path=img_path,
        save_path=save_path,
        title=f'Grad-CAM — {CLASS_NAMES[cls]} (класс {cls})'
    )
    print()

## Шаг 6 — Сводный плакат для диплома (все классы на одном рисунке)

In [ ]:
import math

n     = len(samples)
ncols = 4
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4.5))
fig.suptitle(
    'Grad-CAM визуализация — EfficientNet-B0 (ODIR-5K)\n'
    'Тепловая карта показывает область принятия решения модели',
    fontsize=13, y=1.01
)
axes = axes.flatten()

for ax_idx, (cls, img_path) in enumerate(samples.items()):
    orig_img = np.array(Image.open(img_path).convert('RGB'))
    tensor   = transforms(image=orig_img)['image'].unsqueeze(0).to(DEVICE)

    # Предсказание
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(tensor)).cpu().numpy()[0]

    # Grad-CAM для истинного класса
    class_idx   = CLASSES.index(cls)
    tensor_grad = tensor.clone().requires_grad_(True)
    cam         = gradcam.generate(tensor_grad, class_idx)
    overlay, _  = overlay_heatmap(orig_img, cam, alpha=0.45)

    ax = axes[ax_idx]
    ax.imshow(overlay)
    prob = probs[class_idx]
    ax.set_title(
        f'{CLASS_NAMES[cls]}\nP = {prob*100:.1f}%',
        fontsize=10,
        color='#185FA5' if prob >= 0.7 else '#BA7517'
    )
    ax.axis('off')

# Скрываем пустые ячейки
for i in range(len(samples), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
poster_path = GRADCAM_DIR / 'gradcam_poster.png'
plt.savefig(poster_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Плакат сохранён → {poster_path}')

## Шаг 7 — Тест на вашем снимке

In [ ]:
from google.colab import files as colab_files

print('Загрузите любой снимок глазного дна...')
uploaded = colab_files.upload()
image_path = list(uploaded.keys())[0]

predict_and_visualize(
    image_path=image_path,
    save_path=GRADCAM_DIR / f'gradcam_custom_{image_path}',
)

## Шаг 8 — Сохранение в Google Drive

In [ ]:
import shutil

gradcam_drive = DRIVE_DIR / 'gradcam'
gradcam_drive.mkdir(parents=True, exist_ok=True)

for f in GRADCAM_DIR.iterdir():
    shutil.copy(f, gradcam_drive / f.name)

print('✅ Все Grad-CAM изображения сохранены в Drive')
print('📁 MyDrive/diploma_odir/gradcam/')
print()
for f in sorted(gradcam_drive.iterdir()):
    size = f.stat().st_size / 1e3
    print(f'  {f.name:<45} ({size:.0f} KB)')